# NoisiQ — Week 9 Notebook
**Noise-Aware Quantum Circuit Simulation and Visualization**
*Period: May 18 – May 24, 2026*

---

## Purpose of This Notebook

This notebook serves as the living record and demonstration for Week 9 of the NoisiQ project.

By the end of this notebook you should be able to:
- Apply Hahn echo and XY4 dynamical decoupling sequences to a noisy circuit
- Chain suppression steps using the `SuppressionPipeline`
- Display before/after error heatmap comparisons side-by-side
- Plot fidelity decay curves before and after suppression to quantify improvement
- Quantify aggregate error reduction from suppression

---

## Week 9 Milestone Summary

Week 9 adds the **noise suppression module** with dynamical decoupling sequences,
a **suppression pipeline**, and **comparison visualization** showing before/after heatmaps,
delta plots, and aggregate improvement statistics. Also surfaces `plot_fidelity_decay`
(built in Week 7) in a suppression context — before/after fidelity curves are the clearest
summary of how much a DD sequence actually helps.

---

## Status Tracker

| Task | Owner | Status |
|------|-------|--------|
| `noisiq/suppression/__init__.py` — suppression submodule public API | TJ | ☐ Todo |
| `noisiq/suppression/dynamical_decoupling.py` — `HahnEcho`, `XY4Sequence` | TJ | ☐ Todo |
| `noisiq/suppression/pipeline.py` — `SuppressionPipeline` chaining | TJ | ☐ Todo |
| `noisiq/visualization/comparison.py` — `plot_comparison()` side-by-side heatmaps + delta | TJ | ☐ Todo |
| Fidelity decay before/after demo using `run_depth_sweep` + `plot_fidelity_decay` | TJ | ☐ Todo |
| `tests/suppression/test_dynamical_decoupling.py` | TJ | ☐ Todo |
| All tests passing via `pytest` | TJ | ☐ Todo |
| CI passing on GitHub | TJ | ☐ Todo |
| Week 9 demo section of this notebook complete | TJ | ☐ Todo |

---

## File Build Order

```
1. noisiq/suppression/dynamical_decoupling.py  ← DD sequences (depends on ir/circuit)
2. noisiq/suppression/pipeline.py              ← Pipeline (depends on dynamical_decoupling)
3. noisiq/suppression/__init__.py              ← Exposes HahnEcho, XY4Sequence, SuppressionPipeline
4. noisiq/visualization/comparison.py          ← Comparison plots (depends on heatmap)
```

---

## Physics Requirements for Week 9

**Hahn echo:**
- Inserts a single X pulse at the midpoint of an idle window on a target qubit
- Refocuses quasi-static Z noise (low-frequency dephasing)
- Effective when `t_idle >> T2*` but `t_idle << T2`

**XY4 sequence:**
- Inserts X-Y-X-Y pulses at equal intervals in the idle window
- Suppresses both X and Z quasi-static noise
- Requires idle time to be divisible into 4 equal segments

**Expected test outcomes:**
- XY4 applied to a T2-noisy idle qubit reduces dephasing error rate
- Neither DD sequence improves systematic (coherent over-rotation) errors
- DD sequences have no effect when there is no idle time in the circuit

**Comparison plot requirements:**
- Side-by-side heatmaps: left = before suppression, right = after; same color scale on both
- Delta plot (third panel): `after - before` signed difference matrix
- Summary text: mean error probability before vs after; percent reduction

**Fidelity decay comparison:**
- Run `run_depth_sweep` on both original and suppressed circuits
- Plot both decay curves on the same axes (before = dashed, after = solid)
- Area between curves = total suppression benefit

---

## Notes and Decisions Log

| Date | Note | Name |
|------|------|------|
| 2026-05-18 | Week 9 started. Focus on clear before/after visualization as the key deliverable. | DS |
| 2026-04-21 | Added fidelity decay before/after demo to this week — natural fit for suppression comparison. Uses run_depth_sweep + plot_fidelity_decay built in Week 7. | DS |
| | | |

# Installation Instructions (Developer)

```bash
pip install -e .
```

In [ ]:
import noisiq as nq

print(f"noisiq {nq.__version__} imported successfully")

# Build a circuit with significant idle time (where DD helps)
circuit = nq.Circuit(n_qubits=2, name="dd_demo")
circuit.add_gate(nq.ir.H, qubits=[0])
circuit.add_gate(nq.ir.IDLE, qubits=[0], duration=10)
circuit.add_gate(nq.ir.CNOT, qubits=[0, 1])

noise = nq.noise.T2Dephasing(T2=5.0, t_gate=0.5)
noise_config = {i: noise for i in range(len(circuit.operations))}

# 1. Run WITHOUT suppression
print("\n--- Running WITHOUT suppression ---")
runner = nq.backends.ManyShotRunner()
result_before = runner.run(circuit, n_shots=1000, noise_config=noise_config, seed=42)
print(f"Mean error rate (before): {result_before.error_rate_matrix.mean():.4f}")

# 2. Apply XY4 dynamical decoupling
print("\n--- Applying XY4 dynamical decoupling ---")
dd = nq.suppression.XY4Sequence(qubit=0, idle_time=10)
pipeline = nq.suppression.SuppressionPipeline([dd])
suppressed_circuit, metadata = pipeline.apply(circuit)
print(f"Gates added by suppression: {metadata['gates_added']}")

# 3. Run WITH suppression
print("\n--- Running WITH suppression ---")
result_after = runner.run(suppressed_circuit, n_shots=1000, noise_config=noise_config, seed=42)
print(f"Mean error rate (after): {result_after.error_rate_matrix.mean():.4f}")

pct = (result_before.error_rate_matrix.mean() - result_after.error_rate_matrix.mean()) \
      / result_before.error_rate_matrix.mean() * 100
print(f"Error reduction: {pct:.1f}%")

# 4. Side-by-side heatmap comparison + delta
print("\n--- Before/after heatmap comparison ---")
nq.visualization.plot_comparison(
    before=result_before,
    after=result_after,
    circuit=circuit,
    title="XY4 Dynamical Decoupling: T2 Dephasing"
)

# 5. Fidelity decay before/after — clearest summary of suppression benefit
print("\n--- Fidelity decay: before vs after suppression ---")
depth_before = runner.run_depth_sweep(circuit, n_shots=500, noise_config=noise_config, seed=42)
depth_after = runner.run_depth_sweep(suppressed_circuit, n_shots=500, noise_config=noise_config, seed=42)
nq.visualization.charts.plot_fidelity_decay(
    [depth_before, depth_after],
    labels=["Before DD", "After XY4"]
)